In [ ]:
import pandas as pd
import json
import os


# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_FILE  = 'Sample - Superstore.csv'
OUTPUT_DIR  = '../Requirements/environment/input_artifacts'
ORACLE_OUT  = '../Requirements/environment/oracle_expected.json'

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
!pip install faker

# Advanced SwarmBench Data Preparation Script

import pandas as pd
import numpy as np
import random
import os
import json
from faker import Faker

fake = Faker()
random.seed(42)
np.random.seed(42)

# ============================================================
# CONFIG
# ============================================================




REGIONS = ["west", "east", "central", "south"]
QUARTERS = ["q1", "q2", "q3", "q4"]

ROWS_PER_FILE = 1200
DUPLICATE_RATE = 0.08
INVALID_RATE = 0.06

# ============================================================
# COLUMN NAME VARIANTS
# ============================================================

ID_VARIANTS = [
    "row_id",
    "transaction_id",
    "txn_id",
    "sale_id",
    "record_id",
    "id"
]

CATEGORY_VARIANTS = [
    "sub_category",
    "subcategory",
    "sub_cat",
    "product_category",
    "item_type",
    "category"
]

SALES_VARIANTS = [
    "sales",
    "sales_amount",
    "revenue",
    "gross_sales",
    "usd_amount",
    "amount"
]

# ============================================================
# NOISE HELPERS
# ============================================================


def random_sales_format(value):
    """Inject dirty formatting into numeric sales values."""

    formats = [
        lambda x: f"{x:.2f}",
        lambda x: f" ${x:,.2f} ",
        lambda x: f"{x:,.2f}",
        lambda x: f" {x} ",
        lambda x: f"USD {x:.2f}",
    ]

    return random.choice(formats)(value)



INVALID_VALUES = [
    "",
    None,
    "N/A",
    "NULL",
    "-",
    "missing",
    "error",
    "INVALID",
    "-10",
    "-50",
    "-999",
    "USD -45",
    "unknown"
]



def dirty_subcategory(value):
    """Inject casing and whitespace inconsistencies."""

    variants = [
        value,
        value.lower(),
        value.upper(),
        f" {value}",
        f"{value} ",
        f" {value} ",
    ]

    return random.choice(variants)


# ============================================================
# LOAD SOURCE DATA
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading Superstore dataset...")

df = pd.read_csv(INPUT_FILE, encoding="latin1")
print('Regions:', df['Region'].unique()); print('Region counts:\n',
df['Region'].value_counts()); 
print('Total rows:', len(df))

# Normalize source columns
source = pd.DataFrame({
    "row_id": df["Row ID"],
    "sub_category": df["Sub-Category"],
    "sales": df["Sales"]
})

source = source.dropna(subset=["row_id", "sub_category", "sales"])

print(f"Loaded {len(source)} source rows")

# ============================================================
# GENERATE SHARDS
# ============================================================

all_duplicate_ids = []
all_invalid_count = 0
all_duplicate_count = 0

file_metadata = []

for region in REGIONS:
    for quarter in QUARTERS:

        shard_name = f"{region}_{quarter}"
        print(f"Generating {shard_name}...")

        shard = source.sample(ROWS_PER_FILE, replace=True).copy()

        # --------------------------------------------------------
        # Introduce duplicates across files
        # --------------------------------------------------------

        duplicate_count = int(ROWS_PER_FILE * DUPLICATE_RATE)

        if all_duplicate_ids:
            duplicate_ids = random.sample(
                all_duplicate_ids,
                min(len(all_duplicate_ids), duplicate_count)
            )

            for i in range(len(duplicate_ids)):
                shard.iloc[i, shard.columns.get_loc("row_id")] = duplicate_ids[i]

            all_duplicate_count += len(duplicate_ids)

        new_ids = list(shard["row_id"].sample(duplicate_count))
        all_duplicate_ids.extend(new_ids)

        # --------------------------------------------------------
        # Dirty subcategory formatting
        # --------------------------------------------------------

        shard["sub_category"] = shard["sub_category"].apply(dirty_subcategory)

        # --------------------------------------------------------
        # Dirty sales formatting
        # --------------------------------------------------------

        shard["sales"] = shard["sales"].apply(random_sales_format)

        # --------------------------------------------------------
        # Inject invalid rows
        # --------------------------------------------------------

        invalid_count = int(ROWS_PER_FILE * INVALID_RATE)

        invalid_indices = random.sample(range(len(shard)), invalid_count)

        for idx in invalid_indices:
            shard.iloc[idx, shard.columns.get_loc("sales")] = random.choice(INVALID_VALUES)

        all_invalid_count += invalid_count

        # --------------------------------------------------------
        # Randomize schema names
        # --------------------------------------------------------

        id_col = random.choice(ID_VARIANTS)
        category_col = random.choice(CATEGORY_VARIANTS)
        sales_col = random.choice(SALES_VARIANTS)

        renamed = shard.rename(columns={
            "row_id": id_col,
            "sub_category": category_col,
            "sales": sales_col
        })

        # --------------------------------------------------------
        # Add noisy irrelevant columns
        # --------------------------------------------------------

        renamed["customer_notes"] = [fake.sentence() for _ in range(len(renamed))]
        renamed["shipping_comment"] = [fake.text(max_nb_chars=40) for _ in range(len(renamed))]
        renamed["internal_code"] = np.random.randint(1000, 9999, size=len(renamed))
        renamed["processed_by"] = [fake.first_name() for _ in range(len(renamed))]

        # --------------------------------------------------------
        # Shuffle columns randomly
        # --------------------------------------------------------

        cols = list(renamed.columns)
        random.shuffle(cols)
        renamed = renamed[cols]

        # --------------------------------------------------------
        # Shuffle row order
        # --------------------------------------------------------

        renamed = renamed.sample(frac=1).reset_index(drop=True)

        # --------------------------------------------------------
        # Save CSV
        # --------------------------------------------------------

        output_path = os.path.join(
            OUTPUT_DIR,
            f"sales_{region}_{quarter}.csv"
        )

        renamed.to_csv(output_path, index=False)

        file_metadata.append({
            "file": output_path,
            "id_column": id_col,
            "category_column": category_col,
            "sales_column": sales_col,
            "rows": len(renamed)
        })

# ============================================================
# SAVE METADATA
# ============================================================

metadata_path = os.path.join(OUTPUT_DIR, "schema_reference.json")

with open(metadata_path, "w") as f:
    json.dump(file_metadata, f, indent=2)

# ============================================================
# SUMMARY
# ============================================================

print("\n================================================")
print("DATA GENERATION COMPLETE")
print("================================================")
print(f"Generated files: {len(file_metadata)}")
print(f"Approx duplicate rows injected: {all_duplicate_count}")
print(f"Approx invalid rows injected: {all_invalid_count}")
print(f"Output directory: {OUTPUT_DIR}")
print("================================================")


In [4]:
import pandas as pd                                                                                     
import json                                                                                             
import os                                                                                               
                                                                                                          
INPUT_DIR  = '../Requirements/environment/input_artifacts'                                              
ORACLE_OUT = '../Requirements/environment/oracle_expected.json'
SCHEMA_FILE = os.path.join(INPUT_DIR, 'schema_reference.json')

with open(SCHEMA_FILE) as f:
    schema = json.load(f)

frames = []
total_skipped = 0

for entry in schema:
    fname = os.path.basename(entry['file'].replace('\\', '/'))
    region = fname.replace('sales_', '').rsplit('_', 1)[0]

    id_col  = entry['id_column']
    cat_col = entry['category_column']
    sal_col = entry['sales_column']

    d = pd.read_csv(os.path.join(INPUT_DIR, fname))
    d = d.rename(columns={id_col: 'row_id', cat_col: 'sub_category', sal_col: 'sales'})
    d = d[['row_id', 'sub_category', 'sales']].copy()

    d['sales'] = (
        d['sales'].astype(str)
        .str.strip()
        .str.replace('$', '', regex=False)
        .str.replace('USD', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    d['sales'] = pd.to_numeric(d['sales'], errors='coerce')

    invalid = int(d['sales'].isna().sum()) + int((d['sales'].dropna() <= 0).sum())
    total_skipped += invalid

    d = d.dropna(subset=['sales'])
    d = d[d['sales'] > 0]

    d['sub_category'] = d['sub_category'].str.strip().str.title()
    d['region'] = region
    frames.append(d)

combined = pd.concat(frames, ignore_index=True)
before   = len(combined)
combined = combined.drop_duplicates(subset=['row_id'], keep='first')
after    = len(combined)

region_totals = {
    r: round(float(combined[combined['region'] == r]['sales'].sum()), 2)
    for r in ['west', 'east', 'central', 'south']
}
total_sales = round(float(combined['sales'].sum()), 2)
top3 = list(
    combined.groupby('sub_category')['sales']
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index
)

oracle = {
    "total_sales": total_sales,
    "region_totals": region_totals,
    "top_3_sub_categories": top3,
    "duplicate_rows_removed": before - after,
    "records_skipped_missing_sales": total_skipped,
}

print("=== Oracle Expected Output ===")
print(json.dumps(oracle, indent=2))

with open(ORACLE_OUT, 'w') as f:
    json.dump(oracle, f, indent=2)

print(f"\nOracle saved to: {ORACLE_OUT}")

=== Oracle Expected Output ===
{
  "total_sales": 1856319.28,
  "region_totals": {
    "west": 732757.98,
    "east": 575807.4,
    "central": 319601.36,
    "south": 228152.53
  },
  "top_3_sub_categories": [
    "Chairs",
    "Phones",
    "Binders"
  ],
  "duplicate_rows_removed": 9937,
  "records_skipped_missing_sales": 1152
}

Oracle saved to: ../Requirements/environment/oracle_expected.json
